# 09 · Thermal model v0

The chance that a pilot crossing a 1 km cell finds a climb there, from the **terrain**, the **sun** and the **weather**, scored on **sites the model never saw**. One example per crossing (table `crossings` from notebook 08), so busy cells do not look more thermic just because more pilots pass; only what is known when the pilot arrives is used (height above the ground on arrival, never the time spent in the cell).

Models are compared by adding groups of features one at a time: pilot + terrain, then the sun on the slopes, then ERA5 (`era5_single`, and `era5_extra`, `era5_static`, `era5_levels` when notebooks 07c and 07d have built them; run it again after those to see what they add). It also checks the pilots' ceilings against ERA5's boundary layer, and writes the data for the web page.

Writes `database/models/thermal_v0/` (scores, importance, the model, predictions per crossing and per cell) and `database/web_data/`. Use a **High-RAM** runtime; about 30–60 minutes.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
MAX_ROWS = 4_000_000   # crossings sampled for training (all of them fit in High-RAM, but this is plenty)
ROUNDS = 300           # boosting rounds
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "crossings.parquet")), "Run notebook 08 first: database/crossings.parquet is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Train and score

Prints the ceiling check, then the scores of each group of features on held-out sites (AUC: 0.5 is chance, 1 is perfect; skill: how much better than always predicting the average), the calibration (predicted vs observed share, by tenths), the climb-rate model and what the model uses most.

In [ ]:
run(f"{REPO}/scripts/thermal_model.py", DB, "--max-rows", str(MAX_ROWS), "--rounds", str(ROUNDS))

## 4. Charts

In [ ]:
import json, matplotlib.pyplot as plt
M = os.path.join(DB, "models", "thermal_v0")
m = json.load(open(os.path.join(M, "metrics.json")))
cal = pd.DataFrame(m["calibration"])
imp = pd.read_csv(os.path.join(M, "importance.csv")).head(15)[::-1]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot([0, 1], [0, 1], color="#c3c2b7", lw=1)
ax[0].plot(cal["predicted"], cal["observed"], marker="o", color="#2a78d6", lw=2)
ax[0].set(xlabel="predicted chance of a climb", ylabel="observed share", title="Calibration on sites left out")
ax[1].barh(imp["feature"], imp["share"], color="#2a78d6", height=0.6)
ax[1].set(xlabel="share of the model's gain", title="What the model uses most")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
pd.DataFrame(m["climb"]).T[["auc", "log_loss_skill", "features"]].round(3)

## 5. Data for the web page

Sites and, per site, the cells crossed at least 20 times with how often pilots found a climb there and what the model predicts. Places and aggregates only, no pilot names. Written to `database/web_data/`; `web/README.md` says how to publish it.

In [ ]:
run(f"{REPO}/scripts/export_web.py", DB)